
# Multi-Domain Issues to Actions — Domain Shift Benchmark & RAG

**Motivation:** Convert **issues** into **clear actions** with a rigorous, production-minded approach:
- Baselines vs. Embeddings vs. LLM
- Retail→Fintech domain shift
- RAG-grounded, short action plans


## 0) Setup — *Why*: reproducibility and clarity matter for production handoff.

In [ ]:

# !pip install -r requirements.txt
import os, glob, pandas as pd, numpy as np, matplotlib.pyplot as plt
from src.utils.io_utils import load_config
from src.utils.metrics import classification_metrics, confusion_df
from src.models.evaluate import run_all
from src.features.embedder import SBERTEmbedder
from src.rag.build_kb import KBIndex
from src.rag.retrieve import build_kb_payload
from src.rag.actions import recommend_actions

cfg = load_config()
print("Config loaded:", cfg)
print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))


## 1) Data & Domain Variance — *Why*: show performance under shift (Retail→Fintech).

In [ ]:

df = pd.read_csv(cfg["data"]["path"])
src, tgt = cfg["data"]["source_domain"], cfg["data"]["target_domain"]
df_src, df_tgt = df[df.domain==src], df[df.domain==tgt]
print("Counts:", len(df_src), len(df_tgt))
df.groupby(["domain","category"]).size().unstack(fill_value=0).head()


## 2) Baselines & Embeddings — *Why*: quantify lift and robustness, not just intuition.

In [ ]:

summary, latencies, y_pred_tgt, y_pred_knn = run_all(df_src, df_tgt, cfg)
summary


In [ ]:

labels = sorted(df["category"].unique())
cm = confusion_df(df_tgt.category.values, y_pred_knn, labels)
cm


## 3) RAG Actions — *Why*: transform insights into concise, auditable steps.

In [ ]:

kb_files = [(os.path.basename(p), open(p).read()) for p in glob.glob("data/kb/*.txt")]
embedder = SBERTEmbedder(cfg["models"]["embedder"])
kb_texts = [t for _,t in kb_files]; kb_ids = [n for n,_ in kb_files]
kb_index = KBIndex(embedder, kb_texts, kb_ids)

issue = df_tgt.text.values[0]
idxs, _ = kb_index.search(issue, k=cfg["rag"]["top_k"])
snips = [(kb_ids[i], kb_texts[i]) for i in idxs]
print("Issue:", issue)
print("\nActions:\n", recommend_actions(issue, snips))


## 4) Cost, Latency, Governance — *Why*: explicit deployment trade-offs.
- Baseline/Embeddings: ~ms, $0; LLM: seconds + $.
- RAG reduces hallucinations; log prompts/outputs; PII redaction.
- Confidence routing: low-confidence → LLM or human.


## 5) Takeaways — embeddings robust under shift; LLM adds coverage; RAG grounds actions.